<a href="https://colab.research.google.com/github/EnesDemir143/recyclableproject/blob/enes/New_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import tensorflow as tf
import cv2
import imghdr
import os

In [4]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [5]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

print("Path to dataset files:", path)

100%|██████████| 744M/744M [00:03<00:00, 196MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8


In [27]:

data_dir = '/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset'

image_exts = ['jpeg','jpg', 'bmp', 'png']

for image_class in os.listdir(data_dir):
    for image in os.listdir(os.path.join(data_dir, image_class)):
        image_path = os.path.join(data_dir, image_class, image)
        try:
            img = cv2.imread(image_path)
            tip = imghdr.what(image_path)
            if tip not in image_exts:
                print('Image not in ext list {}'.format(image_path))
                os.remove(image_path)
        except Exception as e:
            print('Issue with image {}'.format(image_path))
            # os.remove(image_path)

Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_1433.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2784.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2779.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2184.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_3119.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_1678.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/plastic/plastic_2038.jpg
Image not in ext list /root/.cache/kagglehub/datase

In [28]:
import numpy as np
from matplotlib import pyplot as plt

In [29]:
train_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=(400, 400),
    label_mode='categorical',
    batch_size=64,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training")

val_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=(400, 400),
    label_mode='categorical',
    batch_size=64,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
)


Found 19750 files belonging to 10 classes.
Using 15800 files for training.
Found 19750 files belonging to 10 classes.
Using 3950 files for validation.


In [ ]:
base_model = tf.keras.applications.EfficientNetV2S(include_top=False,
                                                   weights='imagenet',
                                                   input_shape=(400, 400, 3))

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
base_model.summary()

Model: "efficientnetv2-s"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 400, 400, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling (Rescaling)     │ (None, 400, 400, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 200, 200, 24)   │            648 │ rescaling[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 200, 200, 24)   │             96 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 200, 200, 24)   │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_conv      │ (None, 200, 200, 24)   │          5,184 │ stem_activation[0][0]  │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_bn        │ (None, 200, 200, 24)   │             96 │ block1a_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_activati… │ (None, 200, 200, 24)   │              0 │ block1a_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_add (Add)         │ (None, 200, 200, 24)   │              0 │ block1a_project_activ… │
│                           │                        │                │ stem_activation[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_conv      │ (None, 200, 200, 24)   │          5,184 │ block1a_add[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_bn        │ (None, 200, 200, 24)   │             96 │ block1b_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_activati… │ (None, 200, 200, 24)   │              0 │ block1b_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_drop (Dropout)    │ (None, 200, 200, 24)   │              0 │ block1b_project_activ… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_add (Add)         │ (None, 200, 200, 24)   │              0 │ block1b_drop[0][0],    │
│                           │                        │                │ block1a_add[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2a_expand_conv  

 Total params: 20,331,360 (77.56 MB)

 Trainable params: 20,177,488 (76.97 MB)

 Non-trainable params: 153,872 (601.06 KB)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
data_augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"),
                                         tf.keras.layers.RandomRotation(0.2),
                                         tf.keras.layers.RandomZoom(0.2),
                                         tf.keras.layers.RandomHeight(0.2),
                                         tf.keras.layers.RandomWidth(0.2),],
                                         name ="data_augmentation")

In [ ]:
name = 'EfficientNetV2S_garbage_classification'

EfficientNetV2S_model = tf.keras.Sequential([
    tf.keras.Input(shape=(None, None, 3)),
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(train_data.class_names), activation='softmax')
  ],name=name)

In [ ]:
EfficientNetV2S_model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])

In [ ]:
EfficientNetV2S_model.summary()

Model: "EfficientNetV2S_garbage_classification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)       │ (None, None, None, 3)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetv2-s (Functional)        │ (None, None, None, 1280)    │      20,331,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 20,531,434 (78.32 MB)

 Trainable params: 199,818 (780.54 KB)

 Non-trainable params: 20,331,616 (77.56 MB)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,verbose=1)

In [ ]:
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                                              factor=0.2,
                                                              patience=2,
                                                              verbose=1,
                                                              min_lr=0.00001)

check_model = tf.keras.callbacks.ModelCheckpoint('EfficientNetV2S_model.keras',
                                                   monitor="val_accuracy",
                                                   mode="max",
                                                   save_best_only=True)
callback = [early_stop, reduce_learning_rate, check_model]

In [ ]:
import time
start_time = time.time()
EfficientNetV2S_history = EfficientNetV2S_model.fit(train_data,
                                                    epochs=25,
                                                    steps_per_epoch=len(train_data),
                                                    validation_data=val_data,
                                                    validation_steps=len(val_data),
                                                    callbacks=callback)

Epoch 1/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 827s 3s/step - accuracy: 0.7784 - loss: 0.6964 - val_accuracy: 0.9456 - val_loss: 0.1949 - learning_rate: 0.0010
Epoch 2/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 725s 3s/step - accuracy: 0.9107 - loss: 0.2843 - val_accuracy: 0.9466 - val_loss: 0.1743 - learning_rate: 0.0010
Epoch 3/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 665s 3s/step - accuracy: 0.9230 - loss: 0.2353 - val_accuracy: 0.9486 - val_loss: 0.1624 - learning_rate: 0.0010
Epoch 4/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 606s 2s/step - accuracy: 0.9299 - loss: 0.2081 - val_accuracy: 0.9532 - val_loss: 0.1526 - learning_rate: 0.0010
Epoch 5/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 566s 2s/step - accuracy: 0.9372 - loss: 0.1884 - val_accuracy: 0.9537 - val_loss: 0.1491 - learning_rate: 0.0010
Epoch 6/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 541s 2s/step - accuracy: 0.9347 - loss: 0.1835 - val_accuracy: 0.9580 - val_loss: 0.1473 - learning_rate: 0.0010
Epoch 7/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 478s 2s/step - accuracy: 0.9434 - loss: 0.

In [ ]:
end_time = time.time()
training_time = end_time - start_time
print("Total training time: {:.2f} seconds".format(training_time))
EfficientNetV2S_model.save("EfficientNetV2S_model.keras")

Total training time: 10595.30 seconds


In [ ]:
!pip install ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.2 MB/s eta 0:00:00


In [14]:
# Download latest version
path = kagglehub.dataset_download("farzadnekouei/trash-type-image-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/farzadnekouei/trash-type-image-dataset/versions/1


In [21]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomHeight, RandomWidth

predict_model = load_model('/content/EfficientNetV2S_model.keras',custom_objects={'RandomHeight':RandomHeight,'RandomWidth':RandomWidth,'RandomFlip':RandomFlip,'RandomRotation':RandomRotation,'RandomZoom':RandomZoom})


In [18]:
data_path = '/root/.cache/kagglehub/datasets/farzadnekouei/trash-type-image-dataset/versions/1/TrashType_Image_Dataset'

os.listdir(data_path)

['trash', 'metal', 'glass', 'paper', 'plastic', 'cardboard']

In [19]:
from PIL import Image

for types in os.listdir(data_path):
    folder = os.path.join(data_path,types)
    for images in os.listdir(folder):
        image_path = os.path.join(folder,images)
        with Image.open(image_path) as img:
            width, height = img.size
            channels = len(img.getbands())
            print((width,height,channels))

(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 384, 3)
(512, 

In [48]:
test_data = tf.keras.utils.image_dataset_from_directory(
    data_path,
    image_size =(400,400),
    label_mode='categorical',
    batch_size=64,
    seed=42,
    shuffle=True
)



Found 2527 files belonging to 6 classes.


In [42]:
test_loss, test_accuracy = predict_model.evaluate(test_data, verbose=0)


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
print("Test Loss: {:.5f}".format(test_loss))
print("Test Accuracy: {:.2f}%".format(test_accuracy * 100))